# 第71章 子图、控件与导出

组合子图、按钮、下拉菜单、范围控件和HTML导出，形成完整交互视图。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。


## 适用场景

多个相关图表需要在一个Figure中协调展示或切换。

## 数据结构

共享维度的多组数据；导出前应控制Trace数量。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 updatemenus 的 direction 从 "down" 改为 "right"，观察按钮排列方向的变化
2. 修改 rangeslider_visible 从 True 为 False，对比有无范围滑块的交互差异
3. 将 to_html 的 include_plotlyjs 从 "cdn" 改为 True，说明内联脚本对HTML文件大小的影响


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rng = np.random.default_rng(55)
monthly = pd.DataFrame({
    "month": ["1月", "2月", "3月", "4月", "5月", "6月"],
    "sales": [120, 148, 139, 176, 205, 228],
    "profit": [18, 24, 21, 32, 39, 47],
    "orders": [1180, 1320, 1280, 1490, 1680, 1850],
})
regional = pd.DataFrame({
    "region": ["华东", "华南", "华北", "西南"] * 2,
    "channel": ["线上"] * 4 + ["线下"] * 4,
    "sales": [86, 72, 64, 48, 42, 35, 38, 31],
})
orders = pd.DataFrame({
    "category": rng.choice(["办公", "数码", "家居"], 220),
    "region": rng.choice(["华东", "华南", "华北"], 220),
    "channel": rng.choice(["自然流量", "广告", "会员"], 220),
    "order_value": np.clip(rng.normal(280, 85, 220), 40, None),
    "items": rng.integers(1, 8, 220),
})
orders["sales"] = orders["order_value"] * orders["items"]
hierarchy = pd.DataFrame({
    "department": ["消费品", "消费品", "科技", "科技", "科技"],
    "category": ["办公", "家居", "手机", "电脑", "配件"],
    "sales": [320, 410, 680, 540, 290],
})
funnel = pd.DataFrame({
    "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
    "users": [12000, 7200, 3100, 1850, 1420],
})
timeline = pd.DataFrame({
    "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
    "start": pd.to_datetime(["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]),
    "finish": pd.to_datetime(["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]),
    "owner": ["数据", "分析", "分析", "负责人"],
})
countries = pd.DataFrame({
    "country": ["CHN", "USA", "JPN", "DEU", "AUS"],
    "market": ["中国", "美国", "日本", "德国", "澳大利亚"],
    "sales": [920, 680, 430, 360, 240],
    "growth": [0.18, 0.11, 0.09, 0.07, 0.14],
})
print("Plotly示例数据已准备")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["销售趋势", "区域销售"])
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["sales"], mode="lines+markers", name="销售额"), row=1, col=1)
totals = regional.groupby("region", as_index=False)["sales"].sum()
fig.add_trace(go.Bar(x=totals["region"], y=totals["sales"], name="区域合计"), row=1, col=2)
fig.update_layout(title="经营分析组合图", template="plotly_white")
fig.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["sales"], mode="lines+markers", name="销售额", visible=True))
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["profit"], mode="lines+markers", name="利润", visible=False))
fig.update_layout(
    title="指标切换",
    updatemenus=[{
        "buttons": [
            {"label": "销售额", "method": "update", "args": [{"visible": [True, False]}, {"title": "月度销售额"}]},
            {"label": "利润", "method": "update", "args": [{"visible": [False, True]}, {"title": "月度利润"}]},
        ],
        "direction": "down",
    }],
    template="plotly_white",
)
fig.show()


## 3. 参数说明

- make_subplots：布局
- updatemenus：按钮
- rangeslider：范围滑块
- to_html：导出


## 4. 结果解读

控件应解决明确任务，默认状态必须可读，交互变化需要保持单位和标题一致。


## 初学者学习路线

这章建议按照“先观察、再模仿、后修改、最后独立完成”的顺序学习，不必一次记住所有参数。

1. 先阅读任务说明，明确这段代码要回答什么问题。
2. 运行一个最小例子，先观察输入、输出和数据形状，再回看每一行代码。
3. 只修改一个参数或一条数据，重新运行并比较前后结果。
4. 完成“综合练习”，最后再看本章小结，把能迁移到其他数据的问题写下来。

运行时如果看到 NameError，通常是前置单元格还没有运行；如果输出和预期不同，先检查变量是否被后面的单元格重新赋值。


## 先做一个小检查

进入正式例子前，先用一句话回答：本章的输入是什么，想得到什么结果？

本章主题是“第71章 子图、控件与导出”。请特别留意三件事：输入的类型或形状、处理中间变量的含义、最后输出能否支持一个清楚的结论。


## Plotly 的学习主线

先完成静态视图 → 增加 hover 明细 → 统一标题和单位 → 增加筛选或按钮 → 调整布局与响应式阅读 → 导出交互结果

每个交互功能都要服务于一个分析问题。先检查默认视图是否可读，再增加交互；悬停提示是补充信息，不能替代坐标轴和标题。


## 本模块练习方式

基础：完成一个可读交互图；提高：增加明细提示；挑战：设计一个筛选或切换控件，并说明它如何减少认知负担。

完成后请写下：输入是什么、处理做了什么、输出说明了什么、还存在什么限制。


## 示例 4：增加 Hover 和统一单位

这一组例子只处理一个小问题。先运行代码，再逐行对照拆解说明。


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.DataFrame({
    "month": ["1月", "2月", "3月"],
    "sales": [120, 150, 180],
    "orders": [12, 15, 18],
})
fig = px.line(df, x="month", y="sales", markers=True, hover_data=["orders"], title="月度销售趋势")
fig.update_layout(xaxis_title="月份", yaxis_title="销售额（万元）")
fig.show()


### 逐步拆解

静态位置回答整体关系，Hover 提供单条记录的明细；标题和单位让图表脱离代码后仍然可读。

建议第一次运行后只改一个输入值，再观察哪一个输出发生变化。


## 示例 5：用按钮切换两个指标

看懂上一个例子后，再观察同一主题在另一种数据或场景中的写法。


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.DataFrame({
    "month": ["1月", "2月", "3月"],
    "sales": [120, 150, 180],
    "orders": [12, 15, 18],
})
fig = px.line(df, x="month", y=["sales", "orders"], markers=True, title="指标切换")
fig.update_layout(
    updatemenus=[{
        "buttons": [
            {"label": "显示销售额", "method": "update", "args": [{"visible": [True, False]}]},
            {"label": "显示订单数", "method": "update", "args": [{"visible": [False, True]}]},
        ]
    }]
)
fig.show()


### 逐步拆解

控件本质上是在切换 trace 的可见性；按钮标签必须和当前显示的指标一致。

自我检查：如果把输入数量、类别或参数改成另一组值，代码是否仍然能运行？


## 教学实验：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({
    "region": ["华东", "华南", "华北", "西南"],
    "sales": [320, 250, 280, 190],
})
fig = px.bar(report, x="region", y="sales", title="地区销售额")
fig.show()


### 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
fig.show()


### 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 常见误区

- 控件太多
- 按钮状态与标题不同步
- HTML过大
- 子图图例重复


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。


In [ ]:
fig = px.line(monthly, x="month", y="sales", markers=True, title="可导出的销售趋势")
fig.update_xaxes(rangeslider_visible=True)
html = fig.to_html(include_plotlyjs="cdn", full_html=True)
print(f"HTML字符数: {len(html):,}")
fig.show()


## 本章小结

组合子图、按钮、下拉菜单、范围控件和HTML导出，形成完整交互视图。


### 你已经掌握

- 判断子图、控件与导出的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 多个相关图表需要在一个Figure中协调展示或切换。 |
| 数据结构 | 共享维度的多组数据；导出前应控制Trace数量。 |
| 结果解读 | 控件应解决明确任务，默认状态必须可读，交互变化需要保持单位和标题一致。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `make_subplots` | 布局 |
| `updatemenus` | 按钮 |
| `rangeslider` | 范围滑块 |
| `to_html` | 导出 |


### 需要注意

- 控件太多
- 按钮状态与标题不同步
- HTML过大
- 子图图例重复


### 完成检查

- [ ] 能判断什么问题适合使用子图、控件与导出
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论
